# Assignment 1 - Phân loại đa phương thức (Multimodal Classification)

## 4.3 Với tập dữ liệu đa phương thức

So sánh ba cách tiếp cận:
- **Zero-shot classification**: Sử dụng CLIP pretrained để phân loại mà không cần fine-tune
- **Few-shot classification**: Fine-tune CLIP classification head với một số lượng nhỏ mẫu (freeze backbone)
- **Full fine-tuning**: Fine-tune toàn bộ CLIP model bao gồm cả backbone (unfreeze)

**Dataset**: COCO (Common Objects in Context) - phiên bản công khai, không cần xác thực

**Model**: CLIP (Contrastive Language-Image Pre-training)

## 1. Import Libraries và Setup

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import os
import time
import random
import warnings
from typing import List, Dict, Tuple
from collections import Counter, defaultdict
warnings.filterwarnings('ignore')

# ── Data Manipulation ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from PIL import Image

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# ── Transformers & CLIP ───────────────────────────────────────────────────────
from transformers import CLIPProcessor, CLIPModel
from transformers import get_linear_schedule_with_warmup

# ── Sklearn Metrics ───────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split

# ── COCO Tools ────────────────────────────────────────────────────────────────
import json
from pycocotools.coco import COCO

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ── Device Configuration ──────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Load và Chuẩn bị COCO Dataset (Kaggle)

Sử dụng COCO dataset từ Kaggle. Trên Kaggle, COCO dataset được lưu tại `/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/`.

**Cấu trúc COCO 2017**:
- Train images: `/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017/`
- Validation images: `/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/val2017/`
- Annotations: `/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/`

Chúng ta sẽ tạo classification task dựa trên **tất cả 80 categories** của COCO dataset.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# KAGGLE COCO PATHS - Đường dẫn COCO trên Kaggle
# ═══════════════════════════════════════════════════════════════════════════════
KAGGLE_COCO_ROOT = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017"
TRAIN_IMG_DIR = f"{KAGGLE_COCO_ROOT}/train2017"
VAL_IMG_DIR = f"{KAGGLE_COCO_ROOT}/val2017"
TRAIN_ANN_FILE = f"{KAGGLE_COCO_ROOT}/annotations/instances_train2017.json"
VAL_ANN_FILE = f"{KAGGLE_COCO_ROOT}/annotations/instances_val2017.json"
CAPTIONS_TRAIN_FILE = f"{KAGGLE_COCO_ROOT}/annotations/captions_train2017.json"
CAPTIONS_VAL_FILE = f"{KAGGLE_COCO_ROOT}/annotations/captions_val2017.json"

# Kiểm tra xem có đang chạy trên Kaggle không
IS_KAGGLE = os.path.exists(KAGGLE_COCO_ROOT)

if IS_KAGGLE:
    print("✓ Đang chạy trên Kaggle. Sử dụng Kaggle COCO dataset.")
    print(f"  Root: {KAGGLE_COCO_ROOT}")
else:
    print("⚠️  Không chạy trên Kaggle!")
    print("   Vui lòng cập nhật KAGGLE_COCO_ROOT với đường dẫn COCO local của bạn.")

# ═══════════════════════════════════════════════════════════════════════════════
# TARGET CATEGORIES - TẤT CẢ 80 categories của COCO
# ═══════════════════════════════════════════════════════════════════════════════
TARGET_CATEGORIES = {
    1: 'person', 2: 'bicycle', 3: 'car', 4: 'motorcycle', 5: 'airplane',
    6: 'bus', 7: 'train', 8: 'truck', 9: 'boat', 10: 'traffic light',
    11: 'fire hydrant', 13: 'stop sign', 14: 'parking meter', 15: 'bench', 16: 'bird',
    17: 'cat', 18: 'dog', 19: 'horse', 20: 'sheep', 21: 'cow',
    22: 'elephant', 23: 'bear', 24: 'zebra', 25: 'giraffe', 27: 'backpack',
    28: 'umbrella', 31: 'handbag', 32: 'tie', 33: 'suitcase', 34: 'frisbee',
    35: 'skis', 36: 'snowboard', 37: 'sports ball', 38: 'kite', 39: 'baseball bat',
    40: 'baseball glove', 41: 'skateboard', 42: 'surfboard', 43: 'tennis racket', 44: 'bottle',
    46: 'wine glass', 47: 'cup', 48: 'fork', 49: 'knife', 50: 'spoon',
    51: 'bowl', 52: 'banana', 53: 'apple', 54: 'sandwich', 55: 'orange',
    56: 'broccoli', 57: 'carrot', 58: 'hot dog', 59: 'pizza', 60: 'donut',
    61: 'cake', 62: 'chair', 63: 'couch', 64: 'potted plant', 65: 'bed',
    67: 'dining table', 70: 'toilet', 72: 'tv', 73: 'laptop', 74: 'mouse',
    75: 'remote', 76: 'keyboard', 77: 'cell phone', 78: 'microwave', 79: 'oven',
    80: 'toaster', 81: 'sink', 82: 'refrigerator', 84: 'book', 85: 'clock',
    86: 'vase', 87: 'scissors', 88: 'teddy bear', 89: 'hair drier', 90: 'toothbrush'
}

TARGET_CATEGORY_IDS = list(TARGET_CATEGORIES.keys())
TARGET_CATEGORY_NAMES = list(TARGET_CATEGORIES.values())

print(f"\nCategories mục tiêu: TẤT CẢ {len(TARGET_CATEGORY_NAMES)} categories của COCO")
print(f"10 đầu tiên: {TARGET_CATEGORY_NAMES[:10]}")
print(f"10 cuối cùng: {TARGET_CATEGORY_NAMES[-10:]}")

In [ ]:
def load_coco_kaggle(img_dir, ann_file, captions_file, target_cat_ids, max_samples=5000):
    """
    Load COCO dataset từ cấu trúc thư mục Kaggle.
    
    Args:
        img_dir: Thư mục chứa ảnh
        ann_file: File annotations (instances)
        captions_file: File captions
        target_cat_ids: List các category IDs cần load
        max_samples: Số lượng samples tối đa
    
    Returns:
        filtered_data: List các dict chứa image, caption, category, label
        category_counts: Counter đếm số lượng mỗi category
    """
    print(f"\nĐang load COCO từ Kaggle...")
    print(f"  Images: {img_dir}")
    print(f"  Annotations: {ann_file}")
    print(f"  Captions: {captions_file}")
    
    # Khởi tạo COCO API
    coco = COCO(ann_file)
    coco_caps = COCO(captions_file)
    
    # Lấy tất cả image IDs chứa target categories
    img_ids_set = set()
    for cat_id in target_cat_ids:
        img_ids = coco.getImgIds(catIds=[cat_id])
        img_ids_set.update(img_ids)
    
    print(f"  Tìm thấy {len(img_ids_set)} ảnh chứa target categories")
    
    # Filter và collect samples
    filtered_data = []
    category_counts = Counter()
    samples_per_class = int(max_samples / len(target_cat_ids) * 1.5)
    
    for img_id in img_ids_set:
        if len(filtered_data) >= max_samples:
            break
        
        # Lấy thông tin ảnh
        img_info = coco.loadImgs([img_id])[0]
        img_path = os.path.join(img_dir, img_info['file_name'])
        
        # Kiểm tra ảnh có tồn tại không
        if not os.path.exists(img_path):
            continue
        
        # Lấy annotations cho ảnh này
        ann_ids = coco.getAnnIds(imgIds=[img_id], catIds=target_cat_ids)
        anns = coco.loadAnns(ann_ids)
        
        if not anns:
            continue
        
        # Lấy primary category (object nổi bật nhất theo diện tích)
        primary_cat_id = max(anns, key=lambda x: x.get('area', 0))['category_id']
        
        if primary_cat_id not in target_cat_ids:
            continue
        
        category_name = TARGET_CATEGORIES[primary_cat_id]
        
        # Kiểm tra đã đủ samples cho category này chưa
        if category_counts[category_name] >= samples_per_class:
            continue
        
        # Lấy caption
        cap_ids = coco_caps.getAnnIds(imgIds=[img_id])
        captions = coco_caps.loadAnns(cap_ids)
        caption = captions[0]['caption'] if captions else f"image containing {category_name}"
        
        # Load ảnh
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            continue
        
        # Thêm vào filtered data
        filtered_data.append({
            'image': image,
            'caption': caption,
            'category': category_name,
            'label': TARGET_CATEGORY_NAMES.index(category_name),
            'image_id': img_id,
            'file_name': img_info['file_name']
        })
        
        category_counts[category_name] += 1
        
        if len(filtered_data) % 500 == 0:
            print(f"  Đã thu thập {len(filtered_data)} samples")
            print(f"    Phân phối: {dict(category_counts)}")
    
    print(f"\n✓ Đã load {len(filtered_data)} samples")
    print(f"\nPhân phối category:")
    for cat, count in sorted(category_counts.items(), key=lambda x: -x[1]):
        print(f"  {cat}: {count}")
    
    return filtered_data, category_counts

# ═══════════════════════════════════════════════════════════════════════════════
# LOAD DATASET - Load từ train và val splits
# ═══════════════════════════════════════════════════════════════════════════════
train_data, train_counts = load_coco_kaggle(
    TRAIN_IMG_DIR, TRAIN_ANN_FILE, CAPTIONS_TRAIN_FILE, 
    TARGET_CATEGORY_IDS, max_samples=4000
)

val_data, val_counts = load_coco_kaggle(
    VAL_IMG_DIR, VAL_ANN_FILE, CAPTIONS_VAL_FILE, 
    TARGET_CATEGORY_IDS, max_samples=1500
)

# Gộp train và val datasets
filtered_data = train_data + val_data
category_counts = train_counts + val_counts

print(f"\n✓ Kích thước dataset sau khi gộp: {len(filtered_data)}")
print(f"\nPhân phối cuối cùng:")
for cat, count in sorted(category_counts.items(), key=lambda x: -x[1]):
    print(f"  {cat}: {count}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CHUYỂN ĐỔI SANG DATAFRAME VÀ LỌC CATEGORIES
# ═══════════════════════════════════════════════════════════════════════════════
df = pd.DataFrame(filtered_data)

# Loại bỏ các categories có quá ít samples
min_samples = 100
category_counts_filtered = df['category'].value_counts()
valid_categories = category_counts_filtered[category_counts_filtered >= min_samples].index.tolist()

if len(valid_categories) < len(TARGET_CATEGORY_NAMES):
    print(f"\nĐang lọc các categories có < {min_samples} samples...")
    removed = set(TARGET_CATEGORY_NAMES) - set(valid_categories)
    print(f"Đã loại bỏ: {removed}")
    print(f"Giữ lại {len(valid_categories)} categories: {valid_categories}")
    df = df[df['category'].isin(valid_categories)].reset_index(drop=True)
    TARGET_CATEGORY_NAMES = valid_categories
    # Gán lại labels
    df['label'] = df['category'].apply(lambda x: TARGET_CATEGORY_NAMES.index(x))

print(f"\n✓ Kích thước DataFrame cuối cùng: {df.shape}")
print(f"\nCác cột: {df.columns.tolist()}")
print(f"\nPhân phối label cuối cùng:")
print(df['category'].value_counts())
print(f"\nDữ liệu mẫu:")
print(df[['caption', 'category', 'label']].head(10))

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRỰC QUAN HÓA PHÂN PHỐI DỮ LIỆU
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Biểu đồ cột (Bar plot)
df['category'].value_counts().plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Phân phối theo Class', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Số lượng')
axes[0].tick_params(axis='x', rotation=45)

# Biểu đồ tròn (Pie chart)
df['category'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Phân phối theo Class (%)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

# Phân tích độ dài caption
df['caption_length'] = df['caption'].apply(lambda x: len(str(x).split()))
print(f"\nThống kê độ dài caption:")
print(df['caption_length'].describe())

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# HIỂN THỊ CÁC ẢNH MẪU VỚI CAPTIONS
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

sample_indices = df.sample(min(8, len(df)), random_state=SEED).index
for idx, sample_idx in enumerate(sample_indices):
    img = df.iloc[sample_idx]['image']
    caption = str(df.iloc[sample_idx]['caption'])
    category = df.iloc[sample_idx]['category']
    
    # Chuyển sang RGB nếu cần
    if img.mode != 'RGB':
        img = img.convert('RGB')
    
    axes[idx].imshow(img)
    axes[idx].set_title(f"{category}\n{caption[:50]}...", fontsize=9)
    axes[idx].axis('off')

# Ẩn các subplot trống
for idx in range(len(sample_indices), 8):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 4. Dataset Split

Chia dữ liệu thành:
- **Test set**: 20% (để đánh giá cuối cùng)
- **Zero-shot**: Không dùng training data, chỉ dùng test set
- **Few-shot**: Dùng một số lượng nhỏ samples (k-shot per class) để fine-tune classification head
- **Full fine-tuning**: Dùng nhiều samples hơn (100 per class) để fine-tune toàn bộ model

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CHIA DỮ LIỆU THÀNH TRAIN VÀ TEST
# ═══════════════════════════════════════════════════════════════════════════════
train_df, test_df = train_test_split(
    df, 
    test_size=0.2,  # 20% cho test
    random_state=SEED, 
    stratify=df['label']  # Đảm bảo phân phối đều các class
)

print(f"Kích thước tập train: {len(train_df)}")
print(f"Kích thước tập test: {len(test_df)}")

# ═══════════════════════════════════════════════════════════════════════════════
# TẠO FEW-SHOT DATASETS VỚI CÁC GIÁ TRỊ K KHÁC NHAU
# Few-shot: Chỉ dùng k samples per class để train classification head (freeze backbone)
# ═══════════════════════════════════════════════════════════════════════════════
K_SHOTS = [1, 5, 10, 20]  # Số lượng samples mỗi class

few_shot_datasets = {}
for k in K_SHOTS:
    few_shot_samples = []
    for category in TARGET_CATEGORY_NAMES:
        # Lấy ngẫu nhiên k samples cho mỗi category
        category_samples = train_df[train_df['category'] == category].sample(
            n=min(k, len(train_df[train_df['category'] == category])), 
            random_state=SEED
        )
        few_shot_samples.append(category_samples)
    
    few_shot_df = pd.concat(few_shot_samples).reset_index(drop=True)
    few_shot_datasets[k] = few_shot_df
    print(f"Dataset {k}-shot: {len(few_shot_df)} samples ({k} samples/class)")

# ═══════════════════════════════════════════════════════════════════════════════
# TẠO FULL FINE-TUNING DATASET
# Full fine-tuning: Dùng nhiều data hơn (100 samples/class) để train toàn bộ model (unfreeze backbone)
# ═══════════════════════════════════════════════════════════════════════════════
FULL_FINETUNE_SAMPLES = 100  # 100 samples/class cho full fine-tuning
full_finetune_samples = []
for category in TARGET_CATEGORY_NAMES:
    category_samples = train_df[train_df['category'] == category].sample(
        n=min(FULL_FINETUNE_SAMPLES, len(train_df[train_df['category'] == category])), 
        random_state=SEED
    )
    full_finetune_samples.append(category_samples)

full_finetune_df = pd.concat(full_finetune_samples).reset_index(drop=True)
print(f"\nDataset full fine-tuning: {len(full_finetune_df)} samples ({FULL_FINETUNE_SAMPLES} samples/class)")

print(f"\nPhân phối tập test:")
print(test_df['category'].value_counts())

## 5. Load CLIP Model và Processor

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LOAD CLIP MODEL VÀ PROCESSOR
# ═══════════════════════════════════════════════════════════════════════════════
MODEL_NAME = "openai/clip-vit-base-patch32"

print(f"Đang load CLIP model: {MODEL_NAME}")
clip_model = CLIPModel.from_pretrained(MODEL_NAME)
clip_processor = CLIPProcessor.from_pretrained(MODEL_NAME)

# Chuyển model lên device (GPU hoặc CPU)
clip_model = clip_model.to(device)

# Thông tin model
total_params = sum(p.numel() for p in clip_model.parameters())
trainable_params = sum(p.numel() for p in clip_model.parameters() if p.requires_grad)

print(f"\nModel đã load: {MODEL_NAME}")
print(f"Tổng số parameters: {total_params:,}")
print(f"Parameters có thể train: {trainable_params:,}")

## 6. Custom Dataset Class

In [ ]:
class MultimodalDataset(Dataset):
    """
    Custom Dataset cho CLIP multimodal (image + text).
    Trả về raw image và caption, xử lý trong collate_fn.
    """
    def __init__(self, dataframe, processor, target_categories):
        self.df = dataframe.reset_index(drop=True)
        self.processor = processor
        self.target_categories = target_categories
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = row['image']
        caption = str(row['caption'])
        label = row['label']
        
        # Chuyển sang RGB nếu cần
        if image.mode != 'RGB':
            image = image.convert('RGB')
        
        return {
            'image': image,
            'caption': caption,
            'label': label
        }

def collate_fn(batch, processor):
    """
    Custom collate function để xử lý text và images có độ dài khác nhau.
    
    Xử lý tất cả images và captions cùng lúc để đảm bảo padding đúng.
    Điều này quan trọng vì captions có độ dài khác nhau.
    """
    images = [item['image'] for item in batch]
    captions = [item['caption'] for item in batch]
    labels = torch.tensor([item['label'] for item in batch], dtype=torch.long)
    
    # Xử lý tất cả images và captions cùng lúc để padding đúng
    inputs = processor(
        text=captions,
        images=images,
        return_tensors="pt",
        padding=True,  # Padding captions về cùng độ dài
        truncation=True,  # Truncate nếu quá dài
        max_length=77  # Max length của CLIP
    )
    
    return {
        'input_ids': inputs['input_ids'],
        'attention_mask': inputs['attention_mask'],
        'pixel_values': inputs['pixel_values'],
        'label': labels
    }

# ═══════════════════════════════════════════════════════════════════════════════
# TẠO CUSTOM COLLATE FUNCTION VÀ TEST DATALOADER
# ═══════════════════════════════════════════════════════════════════════════════
from functools import partial
collate_with_processor = partial(collate_fn, processor=clip_processor)

# Tạo test dataset và dataloader
test_dataset = MultimodalDataset(test_df, clip_processor, TARGET_CATEGORY_NAMES)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_with_processor)

print(f"Kích thước test dataset: {len(test_dataset)}")
print(f"Số lượng test batches: {len(test_loader)}")

## 7. Zero-Shot Classification

CLIP được huấn luyện để align image và text representations. Chúng ta có thể sử dụng nó trực tiếp để phân loại bằng cách so sánh image embeddings với text prompt embeddings của các categories.

**Không cần training!** CLIP sử dụng pretrained knowledge để phân loại.

In [ ]:
def zero_shot_classification(model, processor, test_df, target_categories, device):
    """
    Thực hiện zero-shot classification sử dụng CLIP.
    
    Cách hoạt động:
    1. Tạo text prompts cho mỗi category: "a photo of a {category}"
    2. Encode cả image và text prompts qua CLIP
    3. Tính similarity giữa image embedding và text embeddings
    4. Category có similarity cao nhất là prediction
    
    Args:
        model: CLIP model
        processor: CLIP processor
        test_df: Test dataframe
        target_categories: List tên các categories
        device: Device (cuda/cpu)
    
    Returns:
        predictions, labels, probabilities
    """
    model.eval()
    
    # Tạo text prompts cho mỗi category
    text_prompts = [f"a photo of a {category}" for category in target_categories]
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    print("Đang thực hiện zero-shot classification...")
    print(f"Text prompts: {text_prompts[:5]}...")  # Hiển thị 5 prompts đầu
    
    with torch.no_grad():
        for idx, row in test_df.iterrows():
            image = row['image']
            label = row['label']
            
            # Chuyển sang RGB nếu cần
            if image.mode != 'RGB':
                image = image.convert('RGB')
            
            # Xử lý inputs: encode cả image và text prompts
            inputs = processor(
                text=text_prompts,
                images=image,
                return_tensors="pt",
                padding=True
            )
            
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # Lấy model outputs và tính similarity
            outputs = model(**inputs)
            logits_per_image = outputs.logits_per_image  # Similarity scores
            probs = logits_per_image.softmax(dim=1)  # Chuyển thành probabilities
            
            pred = probs.argmax(dim=1).item()  # Lấy category có prob cao nhất
            
            all_preds.append(pred)
            all_labels.append(label)
            all_probs.append(probs.cpu().numpy()[0])
            
            if (idx + 1) % 100 == 0:
                print(f"Đã xử lý {idx + 1}/{len(test_df)} samples")
    
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)

# ═══════════════════════════════════════════════════════════════════════════════
# THỰC HIỆN ZERO-SHOT CLASSIFICATION
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*80)
print("ZERO-SHOT CLASSIFICATION")
print("="*80)

start_time = time.time()
zero_shot_preds, zero_shot_labels, zero_shot_probs = zero_shot_classification(
    clip_model, clip_processor, test_df, TARGET_CATEGORY_NAMES, device
)
zero_shot_time = time.time() - start_time

print(f"\nZero-shot classification hoàn thành trong {zero_shot_time:.2f}s")

### 7.1 Zero-Shot Metrics

In [ ]:
def compute_metrics(y_true, y_pred, target_categories):
    """
    Tính toán các metrics đánh giá phân loại toàn diện.
    
    Bao gồm: accuracy, F1 (macro & weighted), precision, recall, và per-class metrics.
    """
    metrics = {}
    
    # Overall metrics
    metrics['accuracy'] = accuracy_score(y_true, y_pred)
    metrics['f1_macro'] = f1_score(y_true, y_pred, average='macro', zero_division=0)
    metrics['f1_weighted'] = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    metrics['precision_macro'] = precision_score(y_true, y_pred, average='macro', zero_division=0)
    metrics['recall_macro'] = recall_score(y_true, y_pred, average='macro', zero_division=0)
    
    # Per-class metrics
    report = classification_report(
        y_true, y_pred, 
        target_names=target_categories,
        output_dict=True,
        zero_division=0
    )
    
    return metrics, report

# ═══════════════════════════════════════════════════════════════════════════════
# TÍNH TOÁN VÀ HIỂN THỊ METRICS CHO ZERO-SHOT
# ═══════════════════════════════════════════════════════════════════════════════
zero_shot_metrics, zero_shot_report = compute_metrics(
    zero_shot_labels, zero_shot_preds, TARGET_CATEGORY_NAMES
)

print("\n" + "="*80)
print("KẾT QUẢ ZERO-SHOT CLASSIFICATION")
print("="*80)
print(f"\nOverall Metrics:")
print(f"  Accuracy:          {zero_shot_metrics['accuracy']:.4f}")
print(f"  F1 (Macro):        {zero_shot_metrics['f1_macro']:.4f}")
print(f"  F1 (Weighted):     {zero_shot_metrics['f1_weighted']:.4f}")
print(f"  Precision (Macro): {zero_shot_metrics['precision_macro']:.4f}")
print(f"  Recall (Macro):    {zero_shot_metrics['recall_macro']:.4f}")
print(f"  Thời gian inference: {zero_shot_time:.2f}s")

print(f"\nMetrics theo từng class:")
per_class_df = pd.DataFrame(zero_shot_report).T
print(per_class_df[['precision', 'recall', 'f1-score', 'support']].head(len(TARGET_CATEGORY_NAMES)))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFUSION MATRIX CHO ZERO-SHOT
# ═══════════════════════════════════════════════════════════════════════════════
cm_zero = confusion_matrix(zero_shot_labels, zero_shot_preds)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_zero, display_labels=TARGET_CATEGORY_NAMES)
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Zero-Shot Classification - Confusion Matrix', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 8. Few-Shot Classification

Fine-tune CLIP với một số lượng nhỏ samples (k-shot per class).

**Ý tưởng chính**: 
- **Freeze CLIP backbone** (giữ nguyên pretrained knowledge)
- **Train classification head** với k samples/class
- Sử dụng multimodal embeddings (image + text) làm input cho classifier

In [ ]:
class CLIPClassifier(nn.Module):
    """
    CLIP với classification head cho few-shot learning.
    
    Kiến trúc:
    - CLIP encoder (có thể freeze hoặc unfreeze)
    - Classification head (2-layer MLP)
    
    Args:
        freeze_backbone: True = chỉ train head (few-shot)
                        False = train toàn bộ (full fine-tuning)
    """
    def __init__(self, clip_model, num_classes, freeze_backbone=True):
        super().__init__()
        self.clip = clip_model
        
        # Freeze CLIP backbone nếu được chỉ định (few-shot mode)
        if freeze_backbone:
            for param in self.clip.parameters():
                param.requires_grad = False
        # Nếu không freeze = full fine-tuning mode (train toàn bộ)
        
        # Classification head: 2-layer MLP
        hidden_size = self.clip.config.projection_dim  # 512 for CLIP base
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),  # Input: concat(image_embed, text_embed)
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_classes)
        )
    
    def forward(self, input_ids, attention_mask, pixel_values):
        # Lấy CLIP embeddings cho image và text
        outputs = self.clip(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values
        )
        
        # Ghép image và text embeddings
        image_embeds = outputs.image_embeds  # (batch, 512)
        text_embeds = outputs.text_embeds    # (batch, 512)
        combined = torch.cat([image_embeds, text_embeds], dim=1)  # (batch, 1024)
        
        # Phân loại thông qua classification head
        logits = self.classifier(combined)
        return logits

In [ ]:
def train_few_shot(model, train_loader, val_loader, epochs=10, lr=1e-4, device='cuda'):
    """
    Train few-shot classifier.
    
    Training loop cơ bản với:
    - AdamW optimizer
    - Linear warmup scheduler (10% steps)
    - Early stopping based on validation accuracy
    """
    criterion = nn.CrossEntropyLoss()
    # Chỉ optimize các parameters có requires_grad=True
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    
    # Linear warmup scheduler
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),  # 10% warmup
        num_training_steps=total_steps
    )
    
    best_val_acc = 0.0
    best_model_state = None
    history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(epochs):
        # ─── Training Loop ─────────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['label'].to(device)
            
            # Forward pass
            optimizer.zero_grad()
            logits = model(input_ids, attention_mask, pixel_values)
            loss = criterion(logits, labels)
            
            # Backward pass
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            # Track metrics
            train_loss += loss.item()
            preds = logits.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
        
        train_acc = train_correct / train_total if train_total > 0 else 0
        avg_train_loss = train_loss / len(train_loader) if len(train_loader) > 0 else 0
        
        # ─── Validation ────────────────────────────────────────────────────
        val_acc = evaluate_few_shot(model, val_loader, device)
        
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{epochs} - "
              f"Train Loss: {avg_train_loss:.4f}, "
              f"Train Acc: {train_acc:.4f}, "
              f"Val Acc: {val_acc:.4f}")
        
        # Lưu best model (early stopping)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return model, history, best_val_acc

def evaluate_few_shot(model, data_loader, device):
    """
    Đánh giá few-shot classifier trên validation/test set.
    """
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['label'].to(device)
            
            logits = model(input_ids, attention_mask, pixel_values)
            preds = logits.argmax(dim=1)
            
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    return correct / total if total > 0 else 0

def predict_few_shot(model, data_loader, device):
    """
    Lấy predictions từ few-shot classifier cho toàn bộ dataset.
    """
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['label'].to(device)
            
            logits = model(input_ids, attention_mask, pixel_values)
            probs = F.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRAIN FEW-SHOT MODELS VỚI CÁC GIÁ TRỊ K KHÁC NHAU
# Freeze backbone (freeze_backbone=True) - chỉ train classification head
# ═══════════════════════════════════════════════════════════════════════════════
few_shot_results = {}

for k in K_SHOTS:
    print("\n" + "="*80)
    print(f"FEW-SHOT CLASSIFICATION: {k}-SHOT")
    print("="*80)
    
    # Tạo datasets và dataloader
    train_dataset = MultimodalDataset(few_shot_datasets[k], clip_processor, TARGET_CATEGORY_NAMES)
    train_loader = DataLoader(
        train_dataset, 
        batch_size=min(8, len(train_dataset)), 
        shuffle=True, 
        collate_fn=collate_with_processor
    )
    
    print(f"\nĐang train với {len(train_dataset)} samples ({k} samples/class)")
    print(f"Freeze backbone: True (few-shot mode)")
    
    # Tạo model với frozen backbone
    model = CLIPClassifier(
        clip_model, 
        num_classes=len(TARGET_CATEGORY_NAMES),
        freeze_backbone=True  # Few-shot: chỉ train head
    ).to(device)
    
    # Train model
    # k nhỏ → nhiều epochs hơn (20 epochs), learning rate cao hơn (1e-4)
    # k lớn → ít epochs hơn (10 epochs), learning rate thấp hơn (5e-5)
    start_time = time.time()
    trained_model, history, best_val_acc = train_few_shot(
        model, 
        train_loader, 
        test_loader,
        epochs=20 if k <= 5 else 10,
        lr=1e-4 if k <= 5 else 5e-5,
        device=device
    )
    training_time = time.time() - start_time
    
    # Evaluate trên test set
    start_time = time.time()
    preds, labels, probs = predict_few_shot(trained_model, test_loader, device)
    inference_time = time.time() - start_time
    
    # Tính metrics
    metrics, report = compute_metrics(labels, preds, TARGET_CATEGORY_NAMES)
    
    # Lưu kết quả
    few_shot_results[k] = {
        'model': trained_model,
        'history': history,
        'preds': preds,
        'labels': labels,
        'probs': probs,
        'metrics': metrics,
        'report': report,
        'training_time': training_time,
        'inference_time': inference_time
    }
    
    print(f"\nKết quả {k}-Shot:")
    print(f"  Accuracy:          {metrics['accuracy']:.4f}")
    print(f"  F1 (Macro):        {metrics['f1_macro']:.4f}")
    print(f"  F1 (Weighted):     {metrics['f1_weighted']:.4f}")
    print(f"  Thời gian training:  {training_time:.2f}s")
    print(f"  Thời gian inference: {inference_time:.2f}s")

## 9. Full Fine-Tuning Classification

Fine-tune toàn bộ CLIP model (unfreeze backbone) với nhiều samples hơn (100/class).

**Khác biệt với Few-shot**:
- **Few-shot**: `freeze_backbone=True` → Chỉ train head (~262K params)
- **Full fine-tuning**: `freeze_backbone=False` → Train toàn bộ (~151M params)

Full fine-tuning cho phép model học sâu hơn từ dữ liệu nhưng cần nhiều data và thời gian hơn.

In [ ]:
print("\n" + "="*80)
print("FULL FINE-TUNING CLASSIFICATION")
print("="*80)

# ═══════════════════════════════════════════════════════════════════════════════
# TẠO DATALOADER CHO FULL FINE-TUNING
# ═══════════════════════════════════════════════════════════════════════════════
full_finetune_dataset = MultimodalDataset(full_finetune_df, clip_processor, TARGET_CATEGORY_NAMES)
full_finetune_train_loader = DataLoader(
    full_finetune_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_with_processor
)

print(f"\nĐang train với {len(full_finetune_dataset)} samples ({FULL_FINETUNE_SAMPLES} samples/class)")
print(f"Freeze backbone: False (full fine-tuning mode)")

# ═══════════════════════════════════════════════════════════════════════════════
# TẠO MODEL VỚI UNFROZEN BACKBONE
# Sự khác biệt chính: freeze_backbone=False → train toàn bộ model
# ═══════════════════════════════════════════════════════════════════════════════
full_finetune_model = CLIPClassifier(
    clip_model,
    num_classes=len(TARGET_CATEGORY_NAMES),
    freeze_backbone=False  # ⚠️ Điểm khác biệt: unfreeze backbone
).to(device)

# Đếm số trainable parameters
trainable_params_full = sum(p.numel() for p in full_finetune_model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params_full:,}")

# ═══════════════════════════════════════════════════════════════════════════════
# TRAIN FULL FINE-TUNING MODEL
# Ít epochs hơn (5) và learning rate thấp hơn (2e-5) cho stability
# ═══════════════════════════════════════════════════════════════════════════════
start_time = time.time()
full_finetune_trained, full_finetune_history, full_finetune_best_val_acc = train_few_shot(
    full_finetune_model,
    full_finetune_train_loader,
    test_loader,
    epochs=5,   # Ít epochs vì train toàn bộ model
    lr=2e-5,    # Learning rate thấp cho stability
    device=device
)
full_finetune_training_time = time.time() - start_time

# ═══════════════════════════════════════════════════════════════════════════════
# EVALUATE TRÊN TEST SET
# ═══════════════════════════════════════════════════════════════════════════════
start_time = time.time()
full_finetune_preds, full_finetune_labels, full_finetune_probs = predict_few_shot(
    full_finetune_trained, test_loader, device
)
full_finetune_inference_time = time.time() - start_time

# Tính metrics
full_finetune_metrics, full_finetune_report = compute_metrics(
    full_finetune_labels, full_finetune_preds, TARGET_CATEGORY_NAMES
)

print(f"\nKết quả Full Fine-Tuning:")
print(f"  Accuracy:          {full_finetune_metrics['accuracy']:.4f}")
print(f"  F1 (Macro):        {full_finetune_metrics['f1_macro']:.4f}")
print(f"  F1 (Weighted):     {full_finetune_metrics['f1_weighted']:.4f}")
print(f"  Precision (Macro): {full_finetune_metrics['precision_macro']:.4f}")
print(f"  Recall (Macro):    {full_finetune_metrics['recall_macro']:.4f}")
print(f"  Thời gian training:  {full_finetune_training_time:.2f}s")
print(f"  Thời gian inference: {full_finetune_inference_time:.2f}s")

### 9.1 Full Fine-Tuning Confusion Matrix

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TẠO DATAFRAME SO SÁNH CÁC PHƯƠNG PHÁP
# ═══════════════════════════════════════════════════════════════════════════════
comparison_data = []

# Zero-shot
comparison_data.append({
    'Method': 'Zero-Shot',
    'K': 0,
    'Backbone': 'Frozen',
    'Accuracy': zero_shot_metrics['accuracy'],
    'F1 (Macro)': zero_shot_metrics['f1_macro'],
    'F1 (Weighted)': zero_shot_metrics['f1_weighted'],
    'Precision': zero_shot_metrics['precision_macro'],
    'Recall': zero_shot_metrics['recall_macro'],
    'Training Time (s)': 0,
    'Inference Time (s)': zero_shot_time
})

# Few-shot (frozen backbone)
for k, results in few_shot_results.items():
    comparison_data.append({
        'Method': f'{k}-Shot',
        'K': k,
        'Backbone': 'Frozen',
        'Accuracy': results['metrics']['accuracy'],
        'F1 (Macro)': results['metrics']['f1_macro'],
        'F1 (Weighted)': results['metrics']['f1_weighted'],
        'Precision': results['metrics']['precision_macro'],
        'Recall': results['metrics']['recall_macro'],
        'Training Time (s)': results['training_time'],
        'Inference Time (s)': results['inference_time']
    })

# Full fine-tuning (unfrozen backbone)
comparison_data.append({
    'Method': 'Full Fine-tuning',
    'K': FULL_FINETUNE_SAMPLES,
    'Backbone': 'Unfrozen',
    'Accuracy': full_finetune_metrics['accuracy'],
    'F1 (Macro)': full_finetune_metrics['f1_macro'],
    'F1 (Weighted)': full_finetune_metrics['f1_weighted'],
    'Precision': full_finetune_metrics['precision_macro'],
    'Recall': full_finetune_metrics['recall_macro'],
    'Training Time (s)': full_finetune_training_time,
    'Inference Time (s)': full_finetune_inference_time
})

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*80)
print("SO SÁNH TỔNG QUAN: ZERO-SHOT vs FEW-SHOT vs FULL FINE-TUNING")
print("="*80)
print(comparison_df.to_string(index=False))

## 10. So sánh Toàn diện: Zero-Shot vs Few-Shot vs Full Fine-Tuning

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFUSION MATRIX CHO FULL FINE-TUNING
# ═══════════════════════════════════════════════════════════════════════════════
cm_full = confusion_matrix(full_finetune_labels, full_finetune_preds)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_full, display_labels=TARGET_CATEGORY_NAMES)
disp.plot(ax=ax, cmap='Oranges', values_format='d')
plt.title('Full Fine-Tuning Classification - Confusion Matrix', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRỰC QUAN HÓA SO SÁNH CÁC PHƯƠNG PHÁP
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Trích xuất dữ liệu để vẽ biểu đồ
methods = comparison_df['Method'].tolist()
accuracies = comparison_df['Accuracy'].tolist()
f1_macro = comparison_df['F1 (Macro)'].tolist()
f1_weighted = comparison_df['F1 (Weighted)'].tolist()
precisions = comparison_df['Precision'].tolist()
recalls = comparison_df['Recall'].tolist()
training_times = comparison_df['Training Time (s)'].tolist()

# 1. So sánh Accuracy
x_pos = np.arange(len(methods))
colors = ['blue' if 'Zero' in m else 'green' if 'Shot' in m else 'orange' for m in methods]
axes[0, 0].bar(x_pos, accuracies, color=colors, alpha=0.7, edgecolor='black')
axes[0, 0].set_xlabel('Phương pháp', fontsize=12)
axes[0, 0].set_ylabel('Accuracy', fontsize=12)
axes[0, 0].set_title('So sánh Accuracy', fontsize=14, fontweight='bold')
axes[0, 0].set_xticks(x_pos)
axes[0, 0].set_xticklabels(methods, rotation=45, ha='right')
axes[0, 0].grid(True, alpha=0.3, axis='y')

# 2. So sánh F1 Score
axes[0, 1].bar(x_pos - 0.2, f1_macro, width=0.4, label='F1 Macro', alpha=0.7, edgecolor='black')
axes[0, 1].bar(x_pos + 0.2, f1_weighted, width=0.4, label='F1 Weighted', alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('Phương pháp', fontsize=12)
axes[0, 1].set_ylabel('F1 Score', fontsize=12)
axes[0, 1].set_title('So sánh F1 Score', fontsize=14, fontweight='bold')
axes[0, 1].set_xticks(x_pos)
axes[0, 1].set_xticklabels(methods, rotation=45, ha='right')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. Precision & Recall
axes[0, 2].bar(x_pos - 0.2, precisions, width=0.4, label='Precision', alpha=0.7, edgecolor='black')
axes[0, 2].bar(x_pos + 0.2, recalls, width=0.4, label='Recall', alpha=0.7, edgecolor='black')
axes[0, 2].set_xlabel('Phương pháp', fontsize=12)
axes[0, 2].set_ylabel('Score', fontsize=12)
axes[0, 2].set_title('So sánh Precision & Recall', fontsize=14, fontweight='bold')
axes[0, 2].set_xticks(x_pos)
axes[0, 2].set_xticklabels(methods, rotation=45, ha='right')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3, axis='y')

# 4. Thời gian Training
axes[1, 0].bar(x_pos, training_times, color=colors, alpha=0.7, edgecolor='black')
axes[1, 0].set_xlabel('Phương pháp', fontsize=12)
axes[1, 0].set_ylabel('Thời gian Training (s)', fontsize=12)
axes[1, 0].set_title('So sánh Thời gian Training', fontsize=14, fontweight='bold')
axes[1, 0].set_xticks(x_pos)
axes[1, 0].set_xticklabels(methods, rotation=45, ha='right')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 5. Accuracy theo K (samples per class)
few_shot_k = [0] + K_SHOTS + [FULL_FINETUNE_SAMPLES]
few_shot_acc = [zero_shot_metrics['accuracy']] + [few_shot_results[k]['metrics']['accuracy'] for k in K_SHOTS] + [full_finetune_metrics['accuracy']]
axes[1, 1].plot(few_shot_k, few_shot_acc, marker='o', linewidth=2, markersize=8, color='darkblue')
axes[1, 1].scatter([0], [zero_shot_metrics['accuracy']], s=150, color='blue', label='Zero-Shot', zorder=5)
axes[1, 1].scatter(K_SHOTS, [few_shot_results[k]['metrics']['accuracy'] for k in K_SHOTS], s=150, color='green', label='Few-Shot (Frozen)', zorder=5)
axes[1, 1].scatter([FULL_FINETUNE_SAMPLES], [full_finetune_metrics['accuracy']], s=150, color='orange', label='Full Fine-tuning (Unfrozen)', zorder=5)
axes[1, 1].set_xlabel('Samples per Class (K)', fontsize=12)
axes[1, 1].set_ylabel('Accuracy', fontsize=12)
axes[1, 1].set_title('Accuracy theo Kích thước Dữ liệu Training', fontsize=14, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# 6. F1 Macro theo K
few_shot_f1 = [zero_shot_metrics['f1_macro']] + [few_shot_results[k]['metrics']['f1_macro'] for k in K_SHOTS] + [full_finetune_metrics['f1_macro']]
axes[1, 2].plot(few_shot_k, few_shot_f1, marker='s', linewidth=2, markersize=8, color='darkgreen')
axes[1, 2].scatter([0], [zero_shot_metrics['f1_macro']], s=150, color='blue', label='Zero-Shot', zorder=5)
axes[1, 2].scatter(K_SHOTS, [few_shot_results[k]['metrics']['f1_macro'] for k in K_SHOTS], s=150, color='green', label='Few-Shot (Frozen)', zorder=5)
axes[1, 2].scatter([FULL_FINETUNE_SAMPLES], [full_finetune_metrics['f1_macro']], s=150, color='orange', label='Full Fine-tuning (Unfrozen)', zorder=5)
axes[1, 2].set_xlabel('Samples per Class (K)', fontsize=12)
axes[1, 2].set_ylabel('F1 Macro', fontsize=12)
axes[1, 2].set_title('F1 Macro theo Kích thước Dữ liệu Training', fontsize=14, fontweight='bold')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Phân tích Hiệu suất theo từng Class

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SO SÁNH F1 SCORE THEO TỪNG CLASS
# ═══════════════════════════════════════════════════════════════════════════════
per_class_comparison = pd.DataFrame({
    'Category': TARGET_CATEGORY_NAMES
})

# Zero-shot
per_class_comparison['Zero-Shot F1'] = [zero_shot_report[cat]['f1-score'] for cat in TARGET_CATEGORY_NAMES]

# Few-shot
for k in K_SHOTS:
    report = few_shot_results[k]['report']
    per_class_comparison[f'{k}-Shot F1'] = [report[cat]['f1-score'] for cat in TARGET_CATEGORY_NAMES]

# Full fine-tuning
per_class_comparison['Full Fine-tune F1'] = [full_finetune_report[cat]['f1-score'] for cat in TARGET_CATEGORY_NAMES]

print("\nSo sánh F1 Score theo từng Class:")
print(per_class_comparison.to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# TRỰC QUAN HÓA HIỆU SUẤT THEO TỪNG CLASS
# ═══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(16, 8))
x = np.arange(len(TARGET_CATEGORY_NAMES))
width = 0.13

ax.bar(x - 2.5*width, per_class_comparison['Zero-Shot F1'], width, label='Zero-Shot', alpha=0.8, color='blue')
for i, k in enumerate(K_SHOTS):
    ax.bar(x + (i-1.5)*width, per_class_comparison[f'{k}-Shot F1'], width, label=f'{k}-Shot', alpha=0.8, color='green')
ax.bar(x + 2.5*width, per_class_comparison['Full Fine-tune F1'], width, label='Full Fine-tune', alpha=0.8, color='orange')

ax.set_xlabel('Category', fontsize=12)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('So sánh F1 Score theo từng Class (Tất cả Phương pháp)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(TARGET_CATEGORY_NAMES, rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 12. So sánh Confusion Matrices

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VẼ CONFUSION MATRICES CHO CÁC PHƯƠNG PHÁP TỐT NHẤT
# ═══════════════════════════════════════════════════════════════════════════════
# Tìm few-shot tốt nhất
best_k = max(K_SHOTS, key=lambda k: few_shot_results[k]['metrics']['accuracy'])

fig, axes = plt.subplots(1, 3, figsize=(24, 7))

# Zero-shot
cm_zero = confusion_matrix(zero_shot_labels, zero_shot_preds)
disp_zero = ConfusionMatrixDisplay(confusion_matrix=cm_zero, display_labels=TARGET_CATEGORY_NAMES)
disp_zero.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title(f'Zero-Shot (Acc: {zero_shot_metrics["accuracy"]:.3f})', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Few-shot tốt nhất
cm_few = confusion_matrix(few_shot_results[best_k]['labels'], few_shot_results[best_k]['preds'])
disp_few = ConfusionMatrixDisplay(confusion_matrix=cm_few, display_labels=TARGET_CATEGORY_NAMES)
disp_few.plot(ax=axes[1], cmap='Greens', values_format='d')
axes[1].set_title(f'{best_k}-Shot Frozen (Acc: {few_shot_results[best_k]["metrics"]["accuracy"]:.3f})', 
                  fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

# Full fine-tuning
cm_full = confusion_matrix(full_finetune_labels, full_finetune_preds)
disp_full = ConfusionMatrixDisplay(confusion_matrix=cm_full, display_labels=TARGET_CATEGORY_NAMES)
disp_full.plot(ax=axes[2], cmap='Oranges', values_format='d')
axes[2].set_title(f'Full Fine-tune Unfrozen (Acc: {full_finetune_metrics["accuracy"]:.3f})', 
                  fontsize=14, fontweight='bold')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 13. Summary và Kết luận

In [ ]:
print("\n" + "="*80)
print("TÓM TẮT VÀ KẾT LUẬN")
print("="*80)

print("\n1. Dataset:")
print(f"   - Tên: COCO 2017 (Đa phương thức: Ảnh + Caption)")
print(f"   - Nguồn: {'Kaggle' if IS_KAGGLE else 'HuggingFace (fallback)'}")
print(f"   - Classes: {len(TARGET_CATEGORY_NAMES)} ({', '.join(TARGET_CATEGORY_NAMES[:5])}...)")
print(f"   - Test samples: {len(test_df)}")

print("\n2. Model:")
print(f"   - Kiến trúc: CLIP (openai/clip-vit-base-patch32)")
print(f"   - Parameters: {total_params:,}")

print("\n3. Ba Phương pháp được So sánh:")
print(f"\n   a) Zero-Shot:")
print(f"      - Không cần training, sử dụng CLIP pretrained trực tiếp")
print(f"      - So sánh ảnh với text prompts cho mỗi category")
print(f"      - Dữ liệu training: 0 samples")
print(f"      - Trainable parameters: 0 (tất cả đều frozen)")

print(f"\n   b) Few-Shot (Frozen Backbone):")
print(f"      - Fine-tune chỉ classification head")
print(f"      - CLIP backbone vẫn frozen (freeze_backbone=True)")
print(f"      - Dữ liệu training: 1-20 samples/class")
print(f"      - Trainable parameters: ~262K (chỉ classification head)")

print(f"\n   c) Full Fine-Tuning (Unfrozen Backbone):")
print(f"      - Fine-tune toàn bộ model bao gồm CLIP backbone")
print(f"      - Tất cả parameters đều trainable (freeze_backbone=False)")
print(f"      - Dữ liệu training: {FULL_FINETUNE_SAMPLES} samples/class")
print(f"      - Trainable parameters: {trainable_params_full:,} (toàn bộ model)")

print("\n4. So sánh Hiệu suất:")
print(f"\n   Zero-Shot:")
print(f"   - Accuracy: {zero_shot_metrics['accuracy']:.4f}")
print(f"   - F1 (Macro): {zero_shot_metrics['f1_macro']:.4f}")
print(f"   - Training: Không cần")
print(f"   - Inference: {zero_shot_time:.2f}s")

for k in K_SHOTS:
    metrics = few_shot_results[k]['metrics']
    print(f"\n   {k}-Shot (Frozen):")
    print(f"   - Accuracy: {metrics['accuracy']:.4f} "
          f"(+{(metrics['accuracy'] - zero_shot_metrics['accuracy'])*100:.2f}%)")
    print(f"   - F1 (Macro): {metrics['f1_macro']:.4f} "
          f"(+{(metrics['f1_macro'] - zero_shot_metrics['f1_macro'])*100:.2f}%)")
    print(f"   - Training: {few_shot_results[k]['training_time']:.2f}s")
    print(f"   - Inference: {few_shot_results[k]['inference_time']:.2f}s")

print(f"\n   Full Fine-Tuning (Unfrozen):")
print(f"   - Accuracy: {full_finetune_metrics['accuracy']:.4f} "
      f"(+{(full_finetune_metrics['accuracy'] - zero_shot_metrics['accuracy'])*100:.2f}%)")
print(f"   - F1 (Macro): {full_finetune_metrics['f1_macro']:.4f} "
      f"(+{(full_finetune_metrics['f1_macro'] - zero_shot_metrics['f1_macro'])*100:.2f}%)")
print(f"   - Training: {full_finetune_training_time:.2f}s")
print(f"   - Inference: {full_finetune_inference_time:.2f}s")

print("\n5. Những Phát hiện Chính:")
print(f"   - Zero-shot CLIP cho baseline tốt mà không cần training")
print(f"   - Few-shot (frozen) cải thiện với dữ liệu và thời gian training tối thiểu")
print(f"   - Full fine-tuning (unfrozen) đạt hiệu suất tốt nhất với nhiều dữ liệu hơn")
print(f"   - Trade-off: Zero-shot (nhanh) vs Full fine-tuning (chính xác)")

# Tìm phương pháp tốt nhất
best_method = comparison_df.loc[comparison_df['Accuracy'].idxmax(), 'Method']
best_acc = comparison_df['Accuracy'].max()
print(f"   - Phương pháp tốt nhất: {best_method} với accuracy {best_acc:.4f}")

improvement_full = (full_finetune_metrics['accuracy'] - zero_shot_metrics['accuracy']) * 100
improvement_best_few = (few_shot_results[best_k]['metrics']['accuracy'] - zero_shot_metrics['accuracy']) * 100

print(f"   - Cải thiện so với zero-shot:")
print(f"     • Few-shot tốt nhất ({best_k}-shot frozen): +{improvement_best_few:.2f}%")
print(f"     • Full fine-tuning (unfrozen): +{improvement_full:.2f}%")

print("\n6. Sự Khác biệt giữa các Phương pháp:")
print(f"   - Zero-shot: Không training, baseline tốt, inference nhanh nhất")
print(f"   - Few-shot: Chỉ train head, cải thiện vừa phải, training nhanh")
print(f"   - Full fine-tuning: Train toàn bộ model, hiệu suất tốt nhất, training chậm hơn")
print(f"   - Điểm khác biệt chính: tham số freeze_backbone (True vs False)")

print("\n7. Khuyến nghị:")
if improvement_full < 5:
    print("   - Zero-shot đã đủ tốt cho task này")
    print("   - Lợi ích từ training là tối thiểu, nên dùng pretrained model")
elif improvement_full < 10:
    print("   - Few-shot learning cung cấp trade-off tốt")
    print("   - Dùng few-shot để triển khai nhanh với cải thiện vừa phải")
else:
    print("   - Full fine-tuning được khuyến nghị để đạt accuracy tốt nhất")
    print("   - Cải thiện đáng kể chứng tỏ chi phí training là xứng đáng")
    print("   - Dùng few-shot nếu thời gian/tài nguyên training bị hạn chế")

print("\n" + "="*80)